#### Embedding层
<font size = 3>

①能否直接使用Tokenizer出来的词元ID作为训练的输入，从而不使用embedding层？（不能）

②能不能使用One-Hot编码以后再输入？（数学上可行，但工程、语义层面强烈不推荐）

③引入了Embedding层以后的作用是什么？
<font>

In [ ]:
#### Embedding层 - 问题①
# 问题：能否直接使用Tokenizer出来的词元ID作为训练的输入，从而不使用embedding层？
# 结论：不能
"""
================================================================================
问题①：为什么不能直接用整数 ID（如 999）作为输入？
================================================================================

【输入】
    一个整数标量，比如 999（代表“猫”）

【内部计算】
    模型第一层：h = W * id + b
    - W 是形状为 (hidden_dim, 1) 的权重矩阵，只有 hidden_dim 个参数
    - 实际计算：h = W · 999

【输出】
    一个形状为 (hidden_dim,) 的向量

【为什么这是错误的】
    1. 方向锁死：
        所有词（id=1、id=999、id=1000）共用同一个 W
        “猫”的方向是 W，“狗”的方向也是 W
        它们只能在同一条直线上伸长或缩短，无法在高维空间中自由分配独立语义方向

    2. 引入虚假数值关系：
        词元ID只是人为分配的编号，本身没有大小语义；
        但这种做法会让模型误以为 1000（狗）比 999（猫）“更大”或“更重要”。

    3. 训练不稳定：
        梯度更新时，ID值会直接乘进梯度里（∂L/∂W = id * ∂L/∂h）。
        编号大的词（如 50000）会让权重 W 的更新幅度远大于编号小的词（如 1），
        导致梯度尺度差异巨大，训练震荡、难以收敛。
================================================================================
"""
import torch
import torch.nn as nn

class BadModel(nn.Module):
    """
    错误做法：直接用整数 ID 作为输入
    """
    def __init__(self, hidden_dim=768):
        super().__init__()
        # 只有一个可训练参数，形状 (hidden_dim, 1)
        # 所有词共用这 hidden_dim 个参数
        self.W = nn.Parameter(torch.randn(hidden_dim, 1))
        self.b = nn.Parameter(torch.randn(hidden_dim))

    def forward(self, token_id):
        # token_id 是一个标量整数，如 999
        # 计算：h = W * id + b
        # 输出永远在 W 的方向上，只是长度随 id 线性变化
        h = torch.mm(self.W, token_id.float().view(1, 1)) + self.b.unsqueeze(1)
        return h.squeeze(1)  # 形状 (hidden_dim,)


if __name__ == "__main__":
    print("=" * 60)
    print("问题①：直接使用整数 ID 作为输入（错误做法）")
    print("=" * 60)

    model = BadModel(hidden_dim=768)
    input_id = torch.tensor(999)   # 输入：标量
    output = model(input_id)       # 输出：形状 (768,)

    print(f"输入: {input_id.item()} (标量)")
    print(f"输出形状: {output.shape}")
    print(f"输出前5个值: {output[:5].detach().numpy().round(4)}")
    print("\n缺陷:")
    print("  - 所有词共用同一个 W，只能在同一方向上伸缩，没有独立语义方向")
    print("  - ID编号大小引入虚假的数值大小关系")
    print("  - 训练时梯度被 ID 值放大，梯度尺度不稳定，容易震荡")


/home/fzus/anaconda3/envs/cs336/bin/python
问题一：直接使用整数 ID 作为输入（错误做法）
输入: 999 (标量)
输出形状: torch.Size([768])
输出前5个值: [ 1135.5175 -1074.7056   388.2506 -1010.9506 -1027.7491]

缺陷:
  - 所有词共用同一个 W，只能在同一方向上伸缩
  - 编号大小引入虚假的数值关系
  - 训练时梯度被 ID 值放大，导致不稳定


In [ ]:
#### Embedding层 - 问题②
# 问题：能不能使用One-Hot编码以后再输入？
# 结论：数学上可行，但工程、语义层面强烈不推荐
"""
================================================================================
问题②：为什么不推荐 One-Hot 编码 + 全连接层？
================================================================================

【输入】
    一个整数 ID（如 999）

【中间转换】
    先构造一个长度为 V（词表大小，假设 50000）的 One-Hot 向量。
    只有第 999 位是 1，其余全是 0。

【内部计算】
    全连接层有一个权重矩阵 W，形状为 (V, hidden_dim)。
    计算 h = One-Hot · W。
    由于 One-Hot 只有一个位置为1，矩阵乘法等价于取出 W 的第 999 行。

【输出】
    一个形状为 (hidden_dim,) 的向量（即 W 的第 999 行）

【为什么这是不推荐的方案】
    ✅ 数学上等价于Embedding查表；❌ 工程、语义层面存在硬伤
    1. 维度爆炸与内存浪费：
        为了取出这一行，需要先生成一个 50000 维的稀疏向量，再做矩阵乘法，额外开销巨大。
        虽然参数量和Embedding一致，但中间向量构造、矩阵乘法非常低效。

    2. 初始语义正交约束：
        任意两个不同词的One-Hot向量点积=0、余弦相似度=0；
        训练初始阶段所有词向量互相正交，模型很难快速学到“猫和狗”比“猫和飞机”更相似这种语义关系。
================================================================================
"""
import torch
import torch.nn as nn

class OneHotModel(nn.Module):
    """
    中间方案：One-Hot 编码 + 全连接层（数学等价Embedding，但工程低效）
    """
    def __init__(self, vocab_size=50000, hidden_dim=768):
        super().__init__()
        # 全连接层，权重形状 (vocab_size, hidden_dim)
        # 参数量 = 50000 × 768 = 3840 万
        self.fc = nn.Linear(vocab_size, hidden_dim, bias=False)

    def forward(self, token_id):
        # 构造 One-Hot 向量，长度 = vocab_size
        one_hot = torch.zeros(vocab_size) # 词表维度大小的全零向量
        one_hot[token_id] = 1.0 # 针对于token_id位置置1，形成One-Hot向量

        # 矩阵乘法，等价于取出权重矩阵的第 token_id 行
        # 但必须先生成 50000 维的稀疏向量，再做大矩阵乘法
        h = self.fc(one_hot)  # 形状 (hidden_dim,)
        return h


if __name__ == "__main__":
    print("=" * 60)
    print("问题②：One-Hot + 全连接层（中间方案，数学等价但低效）")
    print("=" * 60)

    vocab_size = 50000
    model = OneHotModel(vocab_size, hidden_dim=768)
    input_id = torch.tensor(999)
    output = model(input_id)

    print(f"输入: {input_id.item()} (标量 → {vocab_size}维 One-Hot)")
    print(f"参数量: {vocab_size} × 768 = {vocab_size*768:,} 个参数")
    print(f"输出形状: {output.shape}")
    print(f"输出前5个值: {output[:5].detach().numpy().round(4)}")
    print("\n缺陷:")
    print("  - 需要构造超大稀疏向量，额外内存与计算开销")
    print("  - 矩阵乘法替代直接索引，计算效率低")
    print("  - 初始向量强制正交，不利于快速学习语义相似度")


In [ ]:
#### Embedding层 - 问题③
# 问题：引入了Embedding层以后的作用是什么？
# 结论：查表式词向量映射，兼顾效率与语义学习
"""
================================================================================
问题③：Embedding 层是如何正确解决这个问题的？Embedding的作用？
================================================================================

【输入】
    一个整数张量，比如 [999, 1000, 1001]（形状为 (3,)）

【内部机制】
    PyTorch 在底层创建了一个可训练的大矩阵 W_emb，形状为 (V, embed_dim)。
    关键点：它不做矩阵乘法，不构造 One-Hot 向量，而是直接进行索引查表**。
    当输入 999 时，直接按索引读取矩阵的第 999 行数据。
    > 数学上等价于 OneHot+Linear，但底层实现完全不同，没有稀疏向量、没有矩阵乘。

【输出】
    形状为 (3, embed_dim) 的张量，即三个独立的浮点数词向量。

【Embedding层核心作用 & 优势】
    1. 极致高效：
        没有矩阵乘法，不构造中间OneHot向量，就是内存索引查表，速度最快、内存占用最低。

    2. 参数解耦：
        每个词（ID）拥有矩阵中完全独立的一行参数（embed_dim 个专属数字）。
        反向传播更新时，只会更新对应索引那一行，不干扰其他词的向量。

    3. 高维自由度：
        每个词都在 embed_dim 维的空间中拥有独立坐标点。
        通过反向传播，模型可以自由地将语义相近的词（猫、狗）坐标推近，
        语义无关的词（猫、飞机）拉开距离，学习出连续的语义向量空间。
================================================================================
"""
import torch
import torch.nn as nn

class EmbeddingModel(nn.Module):
    """
    正确做法：Embedding 层（查表实现，数学等价OneHot+Linear，工程最优）
    """
    def __init__(self, vocab_size=50000, embed_dim=768):
        super().__init__()
        # Embedding 层，内部是一个形状 (vocab_size, embed_dim) 的可训练矩阵
        # 参数量 = 50000 × 768 = 3840 万（与 One-Hot 方案相同）
        # 但计算方式完全不同：不做乘法，直接索引查表
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim
        )

    def forward(self, token_ids):
        # 直接根据索引查表，不构造 One-Hot，不做矩阵乘法
        # 输入：任意形状的整数张量，如 (batch, seq_len)
        # 输出：形状为 (输入形状, embed_dim) 的浮点数张量
        return self.embedding(token_ids)


if __name__ == "__main__":
    print("=" * 60)
    print("问题③：Embedding 层（查表方案，工业界标准做法）")
    print("=" * 60)

    vocab_size = 50000
    embed_dim = 768
    model = EmbeddingModel(vocab_size, embed_dim)

    # 输入三个词的 ID
    input_ids = torch.tensor([999, 1000, 1001])   # 形状 (3,)
    output = model(input_ids)                     # 形状 (3, embed_dim)

    print(f"输入: {input_ids.tolist()} (整数张量)")
    print(f"参数量: {vocab_size} × {embed_dim} = {vocab_size*embed_dim:,} 个参数")
    print(f"输出形状: {output.shape}")
    print(f"输出前3行，每行前5个值:\n{output[:, :5].detach().numpy().round(4)}")
    print("\n优势:")
    print("  - 直接内存索引查表，无矩阵乘法，计算高效、无稀疏向量开销")
    print("  - 每个词独享一行向量参数，参数解耦，更新互不干扰")
    print("  - 可在高维空间自由学习语义距离，表达词语相似性")
